# Tokenization Logic
In Andrej Karpathy's tutorial, he treats each character as an individual token. However, in practice, tokens include more than just characters. In 3b1b's transformer explanation, he treats a word as an individual token. This is also not entirely representative of what we see in practice.

**(Credit for code, sturcture, and examples: [raiyanyahya](https://github.com/raiyanyahya/how-to-train-your-gpt/blob/master/chapters/02_tokenization.md)! Very fun and helpful)**

### Byte Pair Encoding (BPE)
Consider the word "antidiestablishmentarianism", and all of the subwords contained within it. It certainly seems redundant to represent this entire word as one token, where we can clearly see that it is made up of several smaller words (from "anti" to "antidiestablishmentarian").

The solution we see in practice is BPE, a subword tokenization algorithm used in LLMs like GPT-2/3/4 and Llama to convert text into efficient, numerical inputs. As described by Claude: Start with individual characters as your vocabulary. Repeatedly find the most common pair of adjacent tokens and merge them into a single new token. Repeat until your vocabulary hits a target size.



```
# TRAINING: Analyze our data to learn whcih merges are useful.
start with vocabulary: all unique chars in data
repeat until vocab_size is reached:
  count all adjacent pairs in tokenized data
  find most common pair (eg "t" + "h")
  add merged token to vocabulary
  replace all occurrences of adjacent pair with merged token ("t"+"h" --> "th")
save list of merge rules

# ENCODING: tokenize new text by applying rules developed in training
start with text split into individual chars
apply merge rule in order:
  find all occurrences of particular pair in current tokenization
  merge them
return final list of token ids
```
Note: merge rules are often saved as an ordered list of tuples. Looking at GPT-2's released files on HuggingFace, we see vocab.bpe (merge rules) and encoder.json (token --> id mapping).

### Benefits (directly from OpenAI's documentation)
1. It's reversible and lossless, so you can convert tokens back into the original text
2. It works on arbitrary text, even text that is not in the tokeniser's training data
3. It compresses the text: the token sequence is shorter than the bytes corresponding to the original text. On average, in practice, each token corresponds to about 4 bytes.
4. It attempts to let the model see common subwords. For instance, "ing" is a common subword in English, so BPE encodings will often split "encoding" into tokens like "encod" and "ing" (instead of e.g. "enc" and "oding"). Because the model will then see the "ing" token again and again in different contexts, it helps models generalise and better understand grammar.


In [4]:
from dataclasses import dataclass
import tiktoken
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available:  True
GPU: Tesla T4


In [7]:
from dataclasses import dataclass
import tiktoken # using openai's pre-trained BPE tokenizer

@dataclass
class TokenizerConfig:
  name: str = "gpt2" # GPT-2's tokenizer
  # Vocab_size must match whatever tokenizer we are using; they are coupled
  vocab_size: int = 50257 #50k merges + 256 byte tokens + 1 EOS.

#tiktoken wrapper
class Tokenizer:
  def __init__(self, config: TokenizerConfig = None):
    self.config = config or TokenizerConfig()
    self.enc = tiktoken.get_encoding(self.config.name)
    self.eos_token = "<|endoftext|>"
    self.eos_token_id = self.enc.encode(
        self.eos_token,
        allowed_special = {self.eos_token}
    )[0]

  def encode(self, text:str) -> list[int]:
    return self.enc.encode(text, allowed_special = {self.eos_token})

  def decode(self, ids: list[int]) -> str:
    return self.enc.decode(ids)

  def vocab_size(self) -> int:
    return self.config.vocab_size

In [16]:
def tokenizer_test(text):
  tokenizer = Tokenizer()
  encoded = tokenizer.encode(text)
  decoded = tokenizer.decode(encoded)
  print(f"Original text: {text}")
  print(f"Encoded: {encoded}")
  print(f"Pieces: {[tokenizer.decode([t])for t in encoded]}")
  print(f"Decoded: {decoded}")
  print(f"Match original text: {text == decoded}")

In [17]:
tokenizer_test("Keen to get better at training my models")

Original text: Keen to get better at training my models
Encoded: [42, 6429, 284, 651, 1365, 379, 3047, 616, 4981]
pieces: ['K', 'een', ' to', ' get', ' better', ' at', ' training', ' my', ' models']
Decoded: Keen to get better at training my models
Match original text: True


In [18]:
tokenizer_test(Tokenizer().eos_token)

Original text: <|endoftext|>
Encoded: [50256]
pieces: ['<|endoftext|>']
Decoded: <|endoftext|>
Match original text: True


In [19]:
tokenizer_test("한국어")

Original text: 한국어
Encoded: [47991, 250, 166, 113, 255, 168, 244, 112]
pieces: ['�', '�', '�', '�', '�', '�', '�', '�']
Decoded: 한국어
Match original text: True


In [20]:
tokenizer_test("antidiestablishmentarianism")

Original text: antidiestablishmentarianism
Encoded: [415, 312, 6386, 25380, 3699, 1042]
pieces: ['ant', 'id', 'iest', 'ablishment', 'arian', 'ism']
Decoded: antidiestablishmentarianism
Match original text: True
